## Training notebook

In [1]:
import gym
import highway_env

from stable_baselines3 import PPO
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

%load_ext tensorboard
from datetime import datetime
import os
from ipywidgets import interact
import ipywidgets as widgets
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import CheckpointCallback, EvalCallback
from stable_baselines3.common.env_checker import check_env
from highway_env.tb_callback import TensorboardCallback

from typing import Callable

### Select operating system

In [2]:
os_id = widgets.Text()
def f(desired_os):
    os_id.value = str(desired_os)
interact(f, desired_os=['UBUNTU_PIGO', 'UBUNTU_DELL'])

interactive(children=(Dropdown(description='desired_os', options=('UBUNTU_PIGO', 'UBUNTU_DELL'), value='UBUNTU…

<function __main__.f(desired_os)>

In [3]:
if os_id.value == 'UBUNTU_DELL':
    OUTPUT_PATH = '/home/pighetti/HighwayDRL/training_output'
elif os_id.value == 'UBUNTU_PIGO':
    OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'

### Select environment

In [4]:
env_id = widgets.Text()
def f(input_env):
    env_id.value = str(input_env)
interact(f, input_env=['multipleO-dm-env-v0','highway-v0','dm-env-v0','acc-dm-env-v0','jam-dm-env-v0','overtaken-dm-env-v0','singleO-dm-env-v0'])

interactive(children=(Dropdown(description='input_env', options=('multipleO-dm-env-v0', 'highway-v0', 'dm-env-…

<function __main__.f(input_env)>

In [5]:
total_timesteps = 1e5
# Create and wrap the environment
env = gym.make(env_id.value)
env = Monitor(env, filename='../../training_output/log') 
check_env(env)
env.configure({
    "training_total_timesteps": total_timesteps
})

# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback_CONS')

eval_callback = EvalCallback(env, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [6]:
def linear_schedule(initial_value: float) -> Callable[[float], float]:
    """
    Linear learning rate schedule.

    :param initial_value: Initial learning rate.
    :return: schedule that computes
      current learning rate depending on remaining progress
    """
    def func(progress_remaining: float) -> float:
        """
        Progress will decrease from 1 (beginning) to 0.

        :param progress_remaining:
        :return: current learning rate
        """
        return progress_remaining * initial_value

    return func

In [7]:
ppo_ilr = 3.5e-4
# PPO parameters
model = PPO("MlpPolicy",
        env,
        gamma=0.997,
        batch_size=128,
        n_steps=4096,
        n_epochs=10,
        ent_coef=0.1,
        verbose=0,
        learning_rate=ppo_ilr,
        tensorboard_log=f'{OUTPUT_PATH}/logdir'
        )

In [ ]:
try:
    # Train the agent for n steps
    model.learn(total_timesteps=int(total_timesteps), callback=[checkpoint_callback, eval_callback, tb_callback], progress_bar=True)
finally:
    # Save the trained agent
    model.save(f'{OUTPUT_PATH}/models/ppo_baseline5_sparseonly_{str(os_id.value)}_{total_timesteps:.1E}_{env_id.value}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

## Continued learning

In [4]:
env_cl_id = 'dm-env-v0'
env_cl = gym.make(env_cl_id)
env_cl = Monitor(env_cl, filename='../../training_output/log_cl') 
check_env(env_cl)

OUTPUT_PATH = '/home/pigo/HighwayDRL/training_output'

/home/pigo/miniconda3/envs/highway-env/lib/python3.8/site-packages/stable_baselines3/common/env_checker.py:190: UserWarning: Your observation  has an unconventional shape (neither an image, nor a 1D vector). We recommend you to flatten the observation to have only a 1D vector or use a custom policy to properly process the data.
  warnings.warn(


In [5]:
model_cl = PPO.load(f"{OUTPUT_PATH}/models/ppo_baseline_2lane_sparseonly", env=env_cl, tensorboard_log=f"{OUTPUT_PATH}/logdir")

In [6]:
# Save a checkpoint every 5000 steps
checkpoint_callback = CheckpointCallback(save_freq=5000, save_path=f'{OUTPUT_PATH}/checkpoints',
                                         name_prefix='dm_rl_callback_CL')

eval_callback = EvalCallback(env_cl, best_model_save_path=f'{OUTPUT_PATH}/eval_logs',
                             log_path=f'{OUTPUT_PATH}/eval_logs', eval_freq=1000,
                             deterministic=True, render=False)

tb_callback = TensorboardCallback()

In [7]:
new_timesteps = 1e8
try:
    model_cl.learn(total_timesteps=int(new_timesteps), callback=[checkpoint_callback, eval_callback, tb_callback], reset_num_timesteps=False, progress_bar=True)
finally:
    model_cl.save(f'{OUTPUT_PATH}/models/ppo_no_km_reward_aggressive_{env_cl_id}_{datetime.now().strftime("%d-%m_%H-%M-%S")}.zip')

Output()

Eval num_timesteps=103400, episode_reward=19.85 +/- 43.77

Episode length: 84.60 +/- 30.05

New best mean reward!

Eval num_timesteps=104400, episode_reward=33.24 +/- 26.71

Episode length: 103.80 +/- 17.33

New best mean reward!

Eval num_timesteps=105400, episode_reward=48.03 +/- 42.71

Episode length: 93.60 +/- 33.07

New best mean reward!